# 04 · Approach 2: Bidirectional GRU/LSTM with attention on MFCC sequences
Owner: **Member 2** · Runtime: Colab **GPU**

**Why this model.** A word is an *ordered* sequence of sounds: the same phones in a different order
form a different word. A recurrent network reads the utterance frame by frame and carries a memory
of what came before; gating (GRU/LSTM) keeps that memory stable across the ~100–150 frames of a word
(Hochreiter & Schmidhuber, 1997; Cho et al., 2014). Reading in **both directions** lets each frame be
interpreted with its right context too. **Attention pooling** (Bahdanau et al., 2015; de Andrade et
al., 2018) learns which frames carry the word and down-weights silence and padding, and its weights
can be plotted, which we use in error analysis.
**Limitation:** sequential computation is slow, and long-range memory is weaker than self-attention.

## 0 · Setup
Run this cell first. On **Google Colab** it clones the repo, installs requirements and copies the
data from your Google Drive to the (much faster) local Colab disk. Locally it does nothing special.

**Before the first run:** put `Train.csv`, `Test.csv`, `SampleSubmission.csv` and `Audio.zip`
(downloaded from Zindi) in a Google Drive folder called `swahili_audio`, and set `REPO_URL`.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/YOUR-GROUP/swahili-audio-sequential.git"   # <-- edit once
    REPO_DIR = Path("/content/swahili-audio-sequential")
    DRIVE_DATA = Path("/content/drive/MyDrive/swahili_audio")
    LOCAL_DATA = Path("/content/data")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if not (LOCAL_DATA / ".ready").exists():
        LOCAL_DATA.mkdir(parents=True, exist_ok=True)
        for f in DRIVE_DATA.glob("*.csv"):
            shutil.copy(f, LOCAL_DATA)
        subprocess.run(["unzip", "-q", "-o", str(DRIVE_DATA / "Audio.zip"), "-d", str(LOCAL_DATA)], check=True)
        (LOCAL_DATA / ".ready").touch()
    os.environ["DATA_DIR"] = str(LOCAL_DATA)
    os.environ["CACHE_DIR"] = str(DRIVE_DATA / "cache")          # features persist between sessions
    os.environ["OUTPUT_BACKUP_DIR"] = str(DRIVE_DATA / "outputs")  # results backed up to Drive
else:
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import librosa.display

from src import config as C
from src.data import get_splits, label_encoder, encode, load_unlabelled, add_paths
from src.datasets import ArrayDataset, SpecAugment
from src.features import compute_features, Standardizer, load_audio, logmel, fix_length, n_samples
from src.models import BiRNNAttention
from src.pipeline import run_experiments, finalize
from src.utils import set_seed, get_device

MEMBER, MODEL = "Member 2", "BiRNN attention"
set_seed(C.SEED)
print("Device:", get_device())
df = get_splits()
classes, to_id = label_encoder(df.label)
K = len(classes)
tr, va, te = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
ytr, yva, yte = (encode(d.label, to_id) for d in (tr, va, te))
zindi = add_paths(load_unlabelled())

## Sequence representation: 13 MFCC + Δ + ΔΔ = 39 features per 10 ms frame
MFCCs decorrelate the mel spectrum into a compact vector per frame (Davis & Mermelstein, 1980);
deltas add local dynamics. Log-mel input is tested as an experiment.

In [ ]:
_cache = {}


def features(kind="mfcc"):
    if kind not in _cache:
        X = [compute_features(d.path, kind) for d in (tr, va, te, zindi)]
        sc = Standardizer().fit(X[0])
        _cache[kind] = [sc.transform(x) for x in X]
    return _cache[kind]


print("MFCC sequence shape (features, frames):", features()[0].shape[1:])


def make_datasets(p):
    Xtr, Xva, _, _ = features(p.get("features", "mfcc"))
    aug = SpecAugment(freq_width=4 if p.get("features", "mfcc") == "mfcc" else 8) if p.get("augment") else None
    return ArrayDataset(Xtr, ytr, aug), ArrayDataset(Xva, yva)


def build_model(p):
    n_feat = features(p.get("features", "mfcc"))[0].shape[1]
    return BiRNNAttention(n_feat, K, hidden=p.get("hidden", 128), layers=p.get("layers", 2),
                          rnn=p.get("rnn", "gru"), pooling=p.get("pooling", "attention"),
                          bidirectional=p.get("bidirectional", True), dropout=p.get("dropout", 0.3))

## Experiments

In [ ]:
DEFAULT_TRAIN = dict(epochs=50, lr=2e-3, batch_size=64, weight_decay=1e-4, patience=10)
EXPERIMENTS = [
    dict(id="RNN-01", description="BiGRU (2x128), mean pooling over time",
         hypothesis="Recurrent modelling of frame order beats order-free baselines.",
         motivated_by="Starting point (compare with BL-02 SVM)", params={"pooling": "mean"}),
    dict(id="RNN-02", description="BiGRU + additive attention pooling",
         hypothesis="Attention focuses on the spoken word and ignores silence/padding that mean pooling averages in.",
         motivated_by="RNN-01 + EDA: padding and residual silence make up part of every sequence",
         params={"pooling": "attention"}),
    dict(id="RNN-03", description="BiLSTM + attention",
         hypothesis="LSTM's separate memory cell helps on long sequences; with ~150 frames, GRU may be enough.",
         motivated_by="Gating comparison (Chung et al., 2014 found no universal winner)",
         params={"rnn": "lstm"}),
    dict(id="RNN-04", description="Unidirectional GRU + attention",
         hypothesis="Right-to-left context matters: the end of a word disambiguates its beginning.",
         motivated_by="Ablation of bidirectionality from RNN-02",
         params={"bidirectional": False}),
    dict(id="RNN-05", description="BiGRU + attention + SpecAugment/time shift",
         hypothesis="Augmentation reduces the train/val gap caused by speaker variability.",
         motivated_by="RNN-02 learning curves (overfitting) + EDA within-class variability",
         params={"augment": True}),
    dict(id="RNN-06", description="BiGRU + attention on 64-band log-mel instead of MFCC",
         hypothesis="A learned projection of the full log-mel spectrum retains detail that the 13 MFCCs discard.",
         motivated_by="Representation check: is the MFCC compression limiting?",
         params={"features": "logmel", "augment": True}),
]
best, summary = run_experiments(EXPERIMENTS, build_model, make_datasets, member=MEMBER, model_name=MODEL,
                                n_classes=K, y_train=ytr, default_train=DEFAULT_TRAIN)
summary.round(4)

> Fill the `observation` column in `experiments/log_birnn_attention.csv`: did attention help over mean
> pooling? Did bidirectionality matter? GRU vs LSTM? Add follow-up experiments if a result surprises you.

## Final evaluation on the test split (once)

In [ ]:
kind = best["exp"]["params"].get("features", "mfcc")
_, _, Xte, Xz = features(kind)
model, proba_test, metrics = finalize(best, build_model, ArrayDataset(Xte, yte), yte, te.file.values, classes,
                                      key="birnn_attention", display_name="BiRNN + attention",
                                      zindi_ds=ArrayDataset(Xz), zindi_df=zindi)

## Where does the model look? Attention weights over time
If attention peaks on the voiced part of the word and ignores silence, the pooling does what we
hypothesised. Errors where attention lands on noise are interesting cases for error analysis.

In [ ]:
if best["exp"]["params"].get("pooling", "attention") == "attention":
    dev = get_device()
    model.eval().to(dev)
    pred = proba_test.argmax(1)
    wrong = np.where(pred != yte)[0][:3]
    right = np.where(pred == yte)[0][:3]
    show = list(right) + list(wrong)
    fig, axes = plt.subplots(len(show), 1, figsize=(10, 2.2 * len(show)))
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, show):
        with torch.no_grad():
            model(torch.from_numpy(Xte[i:i + 1]).to(dev))
        a = model.last_attention[0].cpu().numpy()
        S = logmel(fix_length(load_audio(te.path[i]), n_samples()))
        librosa.display.specshow(S, sr=C.SR, hop_length=C.HOP_LENGTH, x_axis="time", ax=ax, cmap="magma")
        ax2 = ax.twinx()
        ax2.plot(np.arange(len(a)) * C.HOP_LENGTH / C.SR, a, color="cyan", lw=1.5)
        ax2.set_yticks([])
        ok = "correct" if pred[i] == yte[i] else "WRONG"
        ax.set_title(f"true '{classes[yte[i]]}' -> predicted '{classes[pred[i]]}' ({ok})", fontsize=9)
        ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(C.RESULTS_DIR / "birnn_attention" / "attention_examples.png", dpi=150)
    plt.show()

In [ ]:
from src.utils import persist_outputs
persist_outputs()